# CSV → SQL Server Loader

A small, reusable pipeline that loads a CSV file into a Microsoft SQL Server
table. It auto-detects the file encoding, loads the data with pandas, then
inserts each row into the database with simple progress tracking.


## IMPORT THE NEEDED LIBRARIES


• pandas for data loading and DataFrame manipulation  
• numpy for numerical operations and data sanitization  
• pyodbc for database connectivity and SQL operations


In [ ]:
import pyodbc
import pandas as pd
import numpy as np

## CHECK ENCODING OF CSV FILE AND LOAD IT

A function that loads the csv file into a pandas data frame while attempting different encodings.

In [ ]:
def read_raw_data(file_path: str) -> pd.DataFrame:
    try:
        encodings = ['utf-8', 'latin1', 'iso-8859-1', 'cp1252', 'windows-1252']

        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, encoding=encoding)
                print(f"Data loaded successfully from {file_path} with encoding: {encoding}")
                return df
            except UnicodeDecodeError:
                continue

        print(f"Failed to load {file_path} with provided encodings")
        return pd.DataFrame()

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()

### Call the function above with the file path as parameter.

In [ ]:
# paste the path of the csv file below
file_path = r"..\data\sample_customers.csv"
df = read_raw_data(file_path)

## CHECK LOADED DATA

Confirm the number of rows and columns loaded

In [ ]:
df.shape

Get a "Snippet" or sample of the loaded data

In [ ]:
df.head()

## CONNECT TO SQL SERVER

Update the connection with the appropriate driver, server and Database.

In [ ]:
# This connection uses Windows authentication
conn = pyodbc.connect(
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=localhost\SQLEXPRESS;"   # update to your server name
    "Database=CSV_Loader_Demo;"        # update to your database name
    "Trusted_Connection=yes;"
)

## INSERT ROWS INTO DATABASE

create cursor object to be used to insert rows using sql

In [ ]:
cursor = conn.cursor()

iterate through each row and insert into the data base,
with tracking logic for every 100 rows inserted (can be changed based on the size of the data being inserted)

In [ ]:
total_rows = len(df)
successful_rows = 0

for index, row in enumerate(df.itertuples(index=False), start=1):

    cursor.execute(
        """
        INSERT INTO [dbo].[customer_age]
            (name, age)
        VALUES (?, ?)
        """,
        row.name,
        row.age
    )

    successful_rows += 1

    if index % 100 == 0:  # show progress every 100 rows
        print(f"Inserting rows {index}/{total_rows}...")

conn.commit()

print(f"Done! Obbie you have Inserted {successful_rows} rows.")

close connection to database

In [ ]:
conn.close()